# Knowledge Tracer — DKT training & visualisation (Colab)

Trains the **Deep Knowledge Tracing** model used by the Knowledge Tracer backend
(format `kt-dkt-v1`: one LSTM layer over concept-weighted inputs, per-problem difficulty
offsets, DKT+ regularisation) and visualises the data, training, calibration and what
the model has learned.

**Data options**
* `synthetic` (default): regenerates the precise synthetic dataset (2500 learners, ~110k
  attempts on the 1000-problem LeetCode catalog) with the repository simulator —
  cohorts, per-concept skills with 1–3 weak concepts, prerequisite coupling, learning,
  forgetting, retries, hints, solutions, abandonment. Ground truth is known, so the
  notebook can measure how well DKT recovers it.
* `upload`: your **real** export (`scripts/export_events.py` → `events.jsonl.gz` + `catalog.json`).

**Runtime:** CPU is fine (~1–3 min of training); a GPU is used automatically if present.
**Output:** `dkt-model.json` — copy it to `model/artifacts/dkt/dkt-model.json` and start the
backend with `KT_DKT_ENABLED=true`.

> Synthetic results measure behaviour under simulator assumptions only; they are not evidence about real learners.

In [ ]:
# Unpack the embedded Knowledge Tracer code + catalog (generated by scripts/build_colab_notebooks.py)
import base64, io, os, pathlib, sys, tarfile
WORK = pathlib.Path(os.environ.get("KT_WORKDIR", "/content/knowledge_tracer" if os.path.isdir("/content") else "kt_colab"))
WORK.mkdir(parents=True, exist_ok=True)
BUNDLE = "..."
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE)), mode="r:gz") as tar:
    tar.extractall(WORK, **({"filter": "data"} if hasattr(tarfile, "data_filter") else {}))
os.chdir(WORK)
for p in (WORK / "model", WORK / "model" / "dkt", WORK / "model" / "evaluation"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
QUICK = os.environ.get("KT_QUICK") == "1"   # smoke-test mode (tiny sizes); leave unset in Colab
print("workdir:", WORK, "| files:", sum(1 for _ in WORK.rglob("*") if _.is_file()), "| quick mode:", QUICK)

In [ ]:
import importlib.util
if importlib.util.find_spec("torch") is None:          # Colab ships torch; local runs may not
    subprocess_args = [sys.executable, "-m", "pip", "install", "-q", "torch", "numpy", "matplotlib"]
    import subprocess; subprocess.run(subprocess_args, check=True)
import json, gzip, math, statistics, collections
import numpy as np, torch, matplotlib
import matplotlib.pyplot as plt
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| numpy", np.__version__)

In [ ]:
import subprocess
def run(*args):
    """Run a bundled script with the notebook's Python and stream its output."""
    env = dict(os.environ, PYTHONPATH=os.pathsep.join(sys.path))
    proc = subprocess.run([sys.executable, *map(str, args)], cwd=WORK, env=env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(proc.stdout[-6000:])
    if proc.returncode != 0:
        raise RuntimeError(f"{args[0]} failed with exit code {proc.returncode}")

## 1. Data
Set `DATA_SOURCE`. For `upload`, you will be asked for `events.jsonl.gz` and `catalog.json`
exported from your backend (learners are keyed pseudonyms).

In [ ]:
DATA_SOURCE = "synthetic"    # "synthetic" | "upload"
N_LEARNERS = 1300 if QUICK else 2500   # train_dkt.py refuses < 1000 learners / 50k attempts
repo_summary = json.loads((WORK / "data/synthetic/v2/summary.json").read_text())   # shipped dataset

if DATA_SOURCE == "synthetic":
    run("scripts/generate_synthetic_dataset.py", "--learners", N_LEARNERS)
    EVENTS = WORK / "data/synthetic/v2/events.jsonl.gz"
    CATALOG = WORK / "data/seed/catalog_1000.json"
    TRAIN_FLAG = "--allow-synthetic"
else:
    from google.colab import files
    up = WORK / "data/real"; up.mkdir(parents=True, exist_ok=True)
    for name, blob in files.upload().items():
        (up / name).write_bytes(blob)
    EVENTS, CATALOG, TRAIN_FLAG = up / "events.jsonl.gz", up / "catalog.json", "--real"

summary = repo_summary if DATA_SOURCE == "synthetic" else None
print("events file:", EVENTS, f"({EVENTS.stat().st_size/1e6:.1f} MB)")

In [ ]:
# Reproducibility check: same simulator + seed as the repository dataset
from kt import events as ev
seqs = ev.load_sequences(EVENTS)
n_events = sum(len(s) for s in seqs.values())
correct = sum(e["correct"] for s in seqs.values() for e in s) / n_events
print(f"learners {len(seqs)} | events {n_events} | correct rate {correct:.4f}")
if summary and not QUICK:
    print(f"repository dataset: events {summary['events']} | correct rate {summary['correctRate']}")
    print("identical to the repository dataset:", summary["events"] == n_events)

In [ ]:
# Dataset overview
catalog = json.loads(CATALOG.read_text())
prior = {p["problemId"]: p.get("priorDifficulty", 5.0) for p in catalog["problems"]}
lengths = [len(s) for s in seqs.values()]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ax[0].hist(lengths, bins=40); ax[0].set(title="attempts per learner", xlabel="attempts", ylabel="learners")
bins = collections.defaultdict(lambda: [0, 0])
for s in seqs.values():
    for e in s:
        if e["attemptNumber"] == 1:
            b = round(prior[e["problemId"]] * 2) / 2
            bins[b][0] += e["correct"]; bins[b][1] += 1
xs = sorted(k for k in bins if bins[k][1] >= 30)
ax[1].plot(xs, [bins[k][0] / bins[k][1] for k in xs], "o-")
ax[1].set(title="first-attempt success vs catalog prior difficulty", xlabel="prior difficulty (0-10)", ylabel="success rate")
results = collections.Counter(e["result"] for s in seqs.values() for e in s)
ax[2].bar(list(results), list(results.values())); ax[2].set_title("outcomes"); ax[2].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()
if DATA_SOURCE == "synthetic":
    cohorts = collections.defaultdict(list)
    for l in ev.read_jsonl(WORK / "data/synthetic/v2/learners.jsonl.gz"):
        s = seqs.get(l["learnerId"], [])
        if s:
            cohorts[l["cohort"]].append(sum(e["correct"] for e in s) / len(s))
    print({c: f"{statistics.fmean(v):.3f} (n={len(v)})" for c, v in cohorts.items()}, "<- success rate by cohort")

## 2. Train DKT
Hyper-parameters are the repository defaults (hidden 64, dropout 0.2, windows of 200
attempts, Adam 1e-3, DKT+ reconstruction/smoothness penalties, early stopping on
validation AUC). Learners are split 70/15/15 by a salted hash, so no learner is in two splits.

In [ ]:
EPOCHS = 3 if QUICK else 40
run("model/dkt/train_dkt.py", TRAIN_FLAG, "--events", EVENTS, "--catalog", CATALOG,
    "--epochs", EPOCHS, "--out", WORK / "model/artifacts/dkt")
metrics = json.loads((WORK / "model/artifacts/dkt/metrics.json").read_text())
print(json.dumps(metrics.get("test", metrics), indent=1))

In [ ]:
history = json.loads((WORK / "model/artifacts/dkt/history.json").read_text())
h = history
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot([r["epoch"] for r in h], [r["trainLoss"] for r in h], "o-"); ax[0].set(title="training loss (DKT+)", xlabel="epoch")
ax[1].plot([r["epoch"] for r in h], [r["val_auc"] for r in h], "o-", color="C1"); ax[1].set(title="validation AUC", xlabel="epoch")
plt.tight_layout(); plt.show()

## 3. Evaluate on held-out learners
Online next-attempt prediction against baselines (IRT with catalog priors, calibrated IRT,
IRT + concept mastery, success rate) and — on synthetic data — the simulator's true
probability ("oracle", the achievable ceiling).

In [ ]:
if DATA_SOURCE == "synthetic":
    run("model/evaluation/evaluate_kt.py", "--events", EVENTS, "--catalog", CATALOG,
        "--dkt", WORK / "model/artifacts/dkt/dkt-model.json", "--out", WORK / "model/reports",
        *(["--max-test-learners", "60"] if QUICK else []))
    report_md = (WORK / "model/reports/kt-evaluation.md").read_text().split("![")[0]
    try:
        from IPython.display import Markdown, Image, display
        display(Markdown(report_md))
        for f in ("calibration.png", "dkt_training.png"):
            display(Image(filename=str(WORK / "model/reports/figures" / f)))
    except ImportError:
        print(report_md)
else:
    print("evaluate_kt.py needs simulator ground truth for the recovery metrics; "
          "the test-split AUC/log-loss above is the real-data result.")

## 4. What did the model learn?
* **Knowledge trajectory** of one held-out learner: DKT concept mastery after every
  attempt (top), next to the simulator's true mastery at the end (synthetic only).
* **Item offsets** δ_q vs. the simulator's true item difficulty.

In [ ]:
from features import Encoder, ReferenceDkt, assisted
from evaluate_kt import NumpyDkt
doc = json.loads((WORK / "model/artifacts/dkt/dkt-model.json").read_text())
dkt, enc = NumpyDkt(doc), Encoder(doc["concepts"], catalog["problems"])
_, _, test_ids = ev.split_learners(seqs)
lid = max(test_ids, key=lambda l: len(seqs[l]))
state, traj = dkt.initial_state(), []
for e in seqs[lid][-200:]:
    state = dkt.step(state, enc.input_vector(e["problemId"], e["correct"], assisted(e)))
    traj.append(dkt.mastery(state))
traj = np.array(traj)
used = [k for k in range(traj.shape[1]) if traj[:, k].std() > 1e-3] or list(range(traj.shape[1]))
fig, ax = plt.subplots(figsize=(13, 4.5))
im = ax.imshow(traj[:, used].T, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_yticks(range(len(used))); ax.set_yticklabels([doc["concepts"][k] for k in used], fontsize=7)
ax.set(title=f"DKT mastery trajectory — learner {lid} ({len(seqs[lid])} attempts)", xlabel="attempt #")
plt.colorbar(im, ax=ax, fraction=0.02); plt.show()
if DATA_SOURCE == "synthetic":
    truth = {l["learnerId"]: l for l in ev.read_jsonl(WORK / "data/synthetic/v2/learners.jsonl.gz")}[lid]
    print("true weak concepts:", truth["weakConcepts"])
    final = dict(zip(doc["concepts"], traj[-1]))
    print("DKT final mastery (lowest 5):", sorted(((round(float(v), 3), c) for c, v in final.items()))[:5])

In [ ]:
if DATA_SOURCE == "synthetic":
    items = {r["problemId"]: r for r in json.loads((WORK / "data/synthetic/v2/items_truth.json").read_text())["items"]}
    offs = doc.get("itemOffsets", {})
    common = [p for p in offs if p in items]
    true_b = np.array([items[p]["trueDifficultyLogit"] for p in common])
    model_b = np.array([doc["weights"]["gamma"] * enc.difficulty[p] + offs[p] for p in common])
    r = np.corrcoef(true_b, model_b)[0, 1]
    plt.figure(figsize=(5, 4.5)); plt.scatter(true_b, model_b, s=6, alpha=0.5)
    plt.xlabel("true item difficulty (logit)"); plt.ylabel("DKT: gamma*d_q + delta_q")
    plt.title(f"item difficulty recovery (r = {r:.3f}, {len(common)} items)"); plt.show()

## 5. Verify & download the model for the backend
The exported JSON is replayed with the pure-Python reference implementation (the same
contract the Java `DktModel` is tested against); predictions must agree to 1e-9.

In [ ]:
ref = ReferenceDkt(doc)
s_ref, s_np, worst = ref.initial_state(), dkt.initial_state(), 0.0
for e in seqs[lid][:25]:
    w, d = enc.weights[e["problemId"]], enc.difficulty[e["problemId"]]
    worst = max(worst, abs(ref.predict(s_ref, w, d, False, e["problemId"]) - dkt.predict(s_np, w, d, False, e["problemId"])))
    x = enc.input_vector(e["problemId"], e["correct"], assisted(e))
    s_ref, s_np = ref.step(s_ref, x), dkt.step(s_np, x)
print(f"max |reference - numpy| over 25 steps: {worst:.2e}")
assert worst < 1e-9
MODEL = WORK / "model/artifacts/dkt/dkt-model.json"
print(f"{MODEL} ({MODEL.stat().st_size/1e3:.0f} kB) trained on {doc.get('trainedOn')}")
try:
    from google.colab import files
    files.download(str(MODEL))
except ImportError:
    pass
print("Backend: copy to model/artifacts/dkt/dkt-model.json and run with KT_DKT_ENABLED=true; "
      "GET /api/models/dkt should report ENABLED.")